# Adivinhar a proxima palavra

In [1]:
import pandas as pd

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

In [3]:
import numpy as np

In [4]:
from string import punctuation

In [5]:
from sklearn.feature_extraction.text import CountVectorizer

In [6]:
from sklearn.model_selection import train_test_split

In [7]:
import random

In [8]:
from sklearn.preprocessing import OneHotEncoder, LabelEncoder

In [11]:
df = pd.read_csv("d:/git/dados/nlp/dialogs.txt", encoding="utf-8",
                 header=None, names=["pessoa_a", "pessoa_b"], sep='\t')

In [12]:
df

,pessoa_a,pessoa_b
0,"hi, how are you doing?",i'm fine. how about yourself?
1,i'm fine. how about yourself?,i'm pretty good. thanks for asking.
2,i'm pretty good. thanks for asking.,no problem. so how have you been?
3,no problem. so how have you been?,i've been great. what about you?
4,i've been great. what about you?,i've been good. i'm in school right now.
...,...,...
3720,that's a good question. maybe it's not old age.,are you right-handed?
3721,are you right-handed?,yes. all my life.
3722,yes. all my life.,you're wearing out your right hand. stop using...
3723,you're wearing out your right hand. stop using...,but i do all my writing with my right hand.


In [19]:
# I´m  => I am    that´s => that is
contraction_dict = {
    "aren't": "are not", "can't": "can not", "could've": "could have",
    "couldn't": "could not", "daren't": "dare not", "didn't": "did not",
    "doesn't": "does not", "don't": "do not", "hadn't": "had not",
    "hasn't": "has not", "haven't": "have not", "he's": "he is",
    "how'd": "how had", "how're": "how are", "how's": "how is",
    "how've": "how have", "i'd": "i had", "i'm": "i am",
    "i've": "i have", "isn't": "is+ not", "it's": "it is",
    "might've": "might have", "mightn't": "might not", "must've": "must have",
    "mustn't": "must not", "needn't": "need not", "oughtn't": "ought not",
    "shan't": "shall not", "she'd": "she had", "she's": "she is",
    "should've": "should have", "shouldn't": "should not", "that'd": "that had",
    "that's": "that is", "there'd": "there had", "there's": "there is",
    "they'd": "they had", "they're": "you are", "they've": "they have",
    "wasn't": "was+ not", "we'd": "we had", "we're": "we are",
    "we've": "we have", "weren't": "were not", "what'd": "what had",
    "what're": "what are", "what's": "what is", "what've": "what have",
    "when'd": "when had", "when're": "when are", "when's": "when is",
    "when've": "when have", "where'd": "where had", "where're": "where are",
    "where's": "where is", "where've": "where have", "who'd": "who had",
    "who're": "who are", "who's": "who is", "who've": "who have",
    "why'd": "why had", "why're": "why are", "why's": "why is",
    "why've": "why have", "would've": "would have", "wouldn't": "would not",
    "you're": "you are", "you've": "you have", "'cause": "because", 
    "ain't": "is not", "aren't": "are not", "can't": "cannot", 
    "could've": "could have", "he's": "he is", "how'll": "how will",
    "i'll": "i will", "it'll": "it will", "it's": "it is", 
    "she'll": "she will", "she's": "she is", "that'll": "that will",
    "there'll": "there will", "they'll": "they will", "they're": "they are",
    "we'll": "we will", "we're": "we are", "what'll": "what will",
    "when'll": "when will", "where'll": "where will", "who'll": "who will",
    "yo're": "you are", "you'll": "you will"
}


In [20]:
table = str.maketrans("", "", punctuation + "\u200b")

def limpar( texto ):
    texto_limpo = texto.lower()
    nova_frase = []
    for token in texto_limpo.split(" "):
        palavra = contraction_dict.get(token, token)
        palavra_limpa = palavra.translate(table)
        nova_frase.append( palavra_limpa )
    return " ".join(nova_frase)

In [21]:
texto = df["pessoa_a"][1]
print("Texto original: ", texto )
print("Texto limpo: ", limpar( texto ) ) 

Texto original:  i'm fine. how about yourself?
Texto limpo:  i am fine how about yourself


In [22]:
df["pessoa_a_limpo"] = df["pessoa_a"].apply(limpar)
df["pessoa_b_limpo"] = df["pessoa_b"].apply(limpar)

In [23]:
df

,pessoa_a,pessoa_b,pessoa_a_limpo,pessoa_b_limpo
0,"hi, how are you doing?",i'm fine. how about yourself?,hi how are you doing,i am fine how about yourself
1,i'm fine. how about yourself?,i'm pretty good. thanks for asking.,i am fine how about yourself,i am pretty good thanks for asking
2,i'm pretty good. thanks for asking.,no problem. so how have you been?,i am pretty good thanks for asking,no problem so how have you been
3,no problem. so how have you been?,i've been great. what about you?,no problem so how have you been,i have been great what about you
4,i've been great. what about you?,i've been good. i'm in school right now.,i have been great what about you,i have been good i am in school right now
...,...,...,...,...
3720,that's a good question. maybe it's not old age.,are you right-handed?,that is a good question maybe it is not old age,are you righthanded
3721,are you right-handed?,yes. all my life.,are you righthanded,yes all my life
3722,yes. all my life.,you're wearing out your right hand. stop using...,yes all my life,you are wearing out your right hand stop using...
3723,you're wearing out your right hand. stop using...,but i do all my writing with my right hand.,you are wearing out your right hand stop using...,but i do all my writing with my right hand


In [44]:
dicionario = {
    "<UNKNOWN>": 0,
    "<PAD>": 1,
    "<BOS>": 2,
    "<EOS>": 3
}

In [45]:
def vectorizer( lista_textos, dicionario, adicionar_fim_de_sentenca = None ):
    lista_numeros = []
    contador_palavras = len(dicionario.keys())
    for texto in lista_textos:
        numeros = []
        frase = texto.split(" ")
        for palavra in frase:
            if palavra not in dicionario: 
                dicionario[palavra] = contador_palavras
                contador_palavras += 1
            numeros.append(dicionario.get(palavra, 0))
        if adicionar_fim_de_sentenca is not None:
            numeros.append(adicionar_fim_de_sentenca)
        lista_numeros.append(numeros)
    return lista_numeros, contador_palavras

In [46]:
pessoa_a_vetorizado, count_palavras = vectorizer( df["pessoa_a_limpo"], dicionario, dicionario["<EOS>"])
print("Palavras no dicionario: ", count_palavras)
print("Pessoa A vetorizado :", pessoa_a_vetorizado[:5])

Palavras no dicionario:  2371
Pessoa A vetorizado : [[4, 5, 6, 7, 8, 3], [9, 10, 11, 5, 12, 13, 3], [9, 10, 14, 15, 16, 17, 18, 3], [19, 20, 21, 5, 22, 7, 23, 3], [9, 22, 23, 24, 25, 12, 7, 3]]


In [47]:
pessoa_b_vetorizado, count_palavras = vectorizer( df["pessoa_b_limpo"], dicionario, dicionario["<EOS>"])
print("Palavras no dicionario: ", count_palavras)
print("Pessoa B vetorizado :", pessoa_b_vetorizado[:5])

Palavras no dicionario:  2498
Pessoa B vetorizado : [[9, 10, 11, 5, 12, 13, 3], [9, 10, 14, 15, 16, 17, 18, 3], [19, 20, 21, 5, 22, 7, 23, 3], [9, 22, 23, 24, 25, 12, 7, 3], [9, 22, 23, 15, 9, 10, 26, 27, 28, 29, 3]]


In [48]:
dicionario_reverso = {}
for chave in dicionario.keys():
    valor = dicionario[chave]
    dicionario_reverso[valor] = chave
dicionario_reverso    

In [67]:
UNKNOWN_INDEX = dicionario["<UNKNOWN>"]
PAD_INDEX = dicionario["<PAD>"]
BOS_INDEX = dicionario["<BOS>"]
EOS_INDEX = dicionario["<EOS>"]


In [49]:
MAX_PALAVRAS = count_palavras
MAX_PALAVRAS

2498

In [50]:
# encoder = LabelEncoder()

In [51]:
# Y_labels = encoder.fit_transform(Y_palavra)
# Y_labels

In [52]:
# print(Y_labels[:20])

In [53]:
# MAX_CLASSES = len(encoder.classes_)
# MAX_CLASSES

In [54]:
# Y = torch.tensor(Y_labels, dtype=torch.long)
# Y.shape

In [55]:
# Identificar qual frase tem a maior quantidade de palavras
max_size = 0
for frase in pessoa_a_vetorizado:
    if len(frase) > max_size:
        max_size = len(frase)
for frase in pessoa_b_vetorizado:
    if len(frase) > max_size:
        max_size = len(frase)        
max_size        

21

In [56]:
def padding(list_numbers, size, padding_value):
    number_pads = size - len(list_numbers)
    frase_padded = []
    for i in range(number_pads):
        frase_padded.append( padding_value )
    frase_padded.extend( list_numbers )
    return frase_padded


In [57]:
pessoa_a_padded = []
for frase in pessoa_a_vetorizado:
    pessoa_a_padded.append( padding(frase, max_size, dicionario["<PAD>"]) )
pessoa_b_padded = []
for frase in pessoa_b_vetorizado:
    pessoa_b_padded.append( padding(frase, max_size, dicionario["<PAD>"]) )    

In [70]:
### 
# [1 , 1 , 1 , 1 , 12, 17, 19, 10]
# [12, 17, 19, 10, 34, 78, 23, 19]
# [1 , 1 , 1 , 1 , 1 , 12, 17, 19]

In [58]:
len(pessoa_b_padded[0])

21

In [59]:
inputs = torch.tensor(pessoa_a_padded, dtype=torch.long)
target = torch.tensor(pessoa_b_padded, dtype=torch.long)

In [60]:
print("inputs: ", inputs.dtype, inputs.shape, inputs.ndim)
print("target: ", target.dtype, target.shape, target.ndim)

inputs:  torch.int64 torch.Size([3725, 21]) 2
target:  torch.int64 torch.Size([3725, 21]) 2


In [62]:
class Encoder( nn.Module ):
    def __init__(self, vocab_size, embedding_size, hidden_dim, padding_idx):
        super().__init__()
        self.embedding = nn.Embedding(
            num_embeddings = vocab_size,
            embedding_dim = embedding_size,
            padding_idx = padding_idx
        )
        self.lstm = nn.LSTM(
            input_size = embedding_size,
            hidden_size = hidden_dim,
            batch_first = True
        )

    def forward(self, input_tensor): 
        embedded = self.embedding( input_tensor )
        outputs, (hidden, cell) = self.lstm( embedded )
        return hidden, cell

In [72]:
class Decoder( nn.Module ):
    def __init__(self, vocab_size, embedding_size, hidden_dim, padding_idx):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings = vocab_size, 
                                      embedding_dim = embedding_size, 
                                      padding_idx = padding_idx)
        self.lstm = nn.LSTM(
            input_size = embedding_size,
            hidden_size = hidden_dim,
            batch_first = True
        )
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input_tokens, hidden, cell):
        embedded = self.embedding( input_tokens )
        output, (hidden, cell) = self.lstm( embedded, (hidden, cell) )
        ultimo_estado = output[-1]
        logits = self.fc(ultimo_estado)
        return logits, hidden, cell

In [75]:
class Seq2Seq( nn.Module ):
    def __init__( self, encoder, decoder, vocab_size ):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.vocab_size = vocab_size

    def forward(self, input_tensor, target_tensor):
        batch_size = input_tensor.shape[0]
        target_len = target_tensor.shape[1]

        hidden, cell = self.encoder( input_tensor )
        #
        # [ [ <BOS> ],
        #   [ <BOS> ], ... batch_size ]
        input_token = torch.full( (batch_size,), BOS_INDEX, dtype=torch.long )
        outputs = torch.zeros( batch_size, target_len, self.vocab_size)
        for i in range(target_len):
            logits, hidden, cell = self.decoder( input_token, hidden, cell )
            outputs[ :, i, : ] = logits
            palavra_prevista = torch.argmax(logits, dim=1)
            input_token = palavra_prevista
        return outputs
            
        

In [76]:
EMBEDDINGS_OUT = 64
HIDDEN_SIZE = 128
EPOCHS = 100

In [77]:
encoder = Encoder( MAX_PALAVRAS, EMBEDDINGS_OUT, HIDDEN_SIZE, PAD_INDEX )
decoder = Decoder( MAX_PALAVRAS, EMBEDDINGS_OUT, HIDDEN_SIZE, PAD_INDEX )
modelo = Seq2Seq( encoder, decoder, MAX_PALAVRAS )
criterio = nn.CrossEntropyLoss() # Cross Entropy Loss
otimizador = optim.Adam( modelo.parameters(), lr=0.01 )

In [79]:
for epoca in range(1, 200):
    modelo.train()
    otimizador.zero_grad()
    outputs = modelo(inputs, target)

    outputs = outputs.reshape( -1, MAX_PALAVRAS )
    target_flat = target.reshape(-1)
    loss = criterio(outputs, target_flat)
    loss.backward()
    otimizador.step()
    
    if epoca % 10 == 0:
        print(f"Epoca: {epoca}\tLoss:{loss}")

RuntimeError: For unbatched 2-D input, hx and cx should also be 2-D but got (3-D, 3-D) tensors

In [73]:
numero_frase = random.randint(0, MAXIMO_FRASES)
texto_completo = df["text_clean"][numero_frase]
texto = " ".join(texto_completo.split(" ")[0:5])
print("Numero da frase : ", numero_frase)
print("Texto escolhido: ", texto)
print("Texto completo: ", texto_completo)

Numero da frase :  979
Texto escolhido:  eu vi muitos filmes na
Texto completo:  eu vi muitos filmes na verdade eu amo filmes de terror b eles são um dos meus gêneros favoritos no entanto este lixo eu me recuso a reconhecer que isso foi dado a honra de filme foi a pior porcaria que eu já tive a tortura de assistir na verdade eu me inscrevi no imdb apenas pelo fato de que eu precisava de uma maneira de pelo menos expressar o quanto esse garbage era horrível eu assisti films eles pelo menos merecem a honra feita em porões de estudantes do ensino médio que foram melhor escritos e dirigidos eu não tenho nada além de pena dos pobres atores neste lixo porque eles estavam apenas tentando ganhar um cheque de pagamento eles terão agora e para sempre essa mancha em seus registros como uma virgem que foi estuprada e recebeu herpes se o escritor  diretor john shiban tiver alguma dignidade depois de obviamente ter falado com inúmeras pessoas para fazer isso ele nunca mais deve se permitir aproximar

In [74]:
# texto = "este é um dos filmes de"
unknown_word = dicionario["<UNKNOWN>"]
texto_em_numeros = []
texto_em_token = texto.split(" ")
for token in texto_em_token: 
    texto_em_numeros.append(dicionario.get(token, unknown_word))
texto_em_numeros

[40, 98, 736, 29, 75]

In [75]:
texto_em_numeros_padded = padding(texto_em_numeros, max_size, padding_idx)
texto_em_numeros_padded

[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 40, 98, 736, 29, 75]

In [76]:
X_predict = torch.tensor([texto_em_numeros_padded], dtype=torch.int32)
X_predict

tensor([[  1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
          40,  98, 736,  29,  75]], dtype=torch.int32)

In [77]:
Y_predict = modelo(X_predict)
numero_palavra = np.argmax(Y_predict.detach().numpy())
encoder.inverse_transform([numero_palavra]) 

array(['minha'], dtype='<U20')